# CARec encoder and dimension study

This notebook will compare general-purpose and instruction-based text encoders as item representations in CARec. We plan to keep the dataset split and recommender setup fixed, then measure warm- and cold-item Recall@10 and NDCG@10 at shared embedding sizes. First, we need to reproduce a baseline.

## References

- [CARec paper, CIKM 2024](https://doi.org/10.1145/3627673.3679535)
- [Official CARec repository](https://github.com/ChenMetanoia/CARec)
- [CARec runner](https://github.com/ChenMetanoia/CARec/blob/main/run_collabcontex.py)
- [CARec setup instructions](https://github.com/ChenMetanoia/CARec#readme)

## Planned sections

1. Set the experiment configuration and random seeds.
2. Load item metadata and the fixed interaction split.
3. Encode item text with the selected general-purpose or instruction-based model.
4. Cache embeddings and reduce them to the target dimensions using the agreed method.
5. Load the item embeddings into CARec and train the recommender.
6. Evaluate warm and cold items, then compare ranking quality, runtime, and storage.

Raw Electronics JSON files are in `data/`. The pinned CARec checkout is in `external/CARec`. Select the project `.venv` interpreter or the NI-PML benchmarks kernel. Run `bash scripts/setup.sh` from the repository root to restore the environment.

## Imports and device

Use Apple's MPS backend when PyTorch reports it available. The fallbacks let the notebook run on another machine too.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

mps_backend = getattr(torch.backends, "mps", None)
if mps_backend is not None and mps_backend.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


## CARec imports

This loads the upstream model and data/evaluation helpers without starting preprocessing or training. The wrapper checks the upstream commit and refuses local source changes.

In [ ]:
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.trainer import Trainer

carec = load_carec()
CollabContex = carec.CollabContex
ColdEvaluator = carec.ColdEvaluator
data_preparation = carec.data_preparation

print(f"CARec: {carec.commit}")